In [ ]:
################################################## RUN THIS NOTEBOOK AND MENTAL ROBERTA ON GOOGLE COLAB ##################################################

In [1]:
!pip install pandas scikit-learn transformers torch datasets google.auth
# environment for Colab

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import pandas as pd

df_distilbert = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/balanced_30k_dataset.csv")
df_distilbert.head()

,title,label
0,Can Mindfulness Make us Happier? Capitalism &a...,0.0
1,It's the small things ....like the Running Xma...,0.0
2,Self-harmed and a little worried about future ...,1.0
3,I dont know that this will end At this point i...,1.0
4,The perfect gift &lt;3,0.0


In [6]:
from sklearn.model_selection import train_test_split

X = df_distilbert["title"]
y = df_distilbert["label"]

X_train_d, X_test_val_d, y_train_d, y_test_val_d = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_test_d, X_val_d, y_test_d, y_val_d = train_test_split(X_test_val_d, y_test_val_d, test_size=0.50, random_state=42, stratify=y_test_val_d)

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
from torch.utils.data import Dataset
from torch import tensor


class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        result = self.tokenizer(self.texts[idx], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

        return {"input_ids": result["input_ids"].squeeze(0), "attention_mask": result["attention_mask"].squeeze(0), "label": tensor(self.labels[idx])}

In [9]:
train_dataset = DepressionDataset(X_train_d.tolist(), y_train_d.tolist(), tokenizer)
test_dataset = DepressionDataset(X_test_d.tolist(), y_test_d.tolist(), tokenizer)
val_dataset = DepressionDataset(X_val_d.tolist(), y_val_d.tolist(), tokenizer)

In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [13]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

In [14]:
######## WARINING USE COLAB FOR TRAINING, THIS CODE IS NOT OPTIMIZED FOR CPU TRAINING ########
model.train()
for epoch in range(3):
    total_loss = 0
    batch_num = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        labels = labels.long()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch + 1}, Batch {batch_num + 1}, Loss: {loss.item():.4f}")
        batch_num += 1

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")

Epoch 1, Batch 1, Loss: 0.6807
Epoch 1, Batch 2, Loss: 0.6975
Epoch 1, Batch 3, Loss: 0.6811
Epoch 1, Batch 4, Loss: 0.7065
Epoch 1, Batch 5, Loss: 0.6380
Epoch 1, Batch 6, Loss: 0.6292
Epoch 1, Batch 7, Loss: 0.6590
Epoch 1, Batch 8, Loss: 0.5778
Epoch 1, Batch 9, Loss: 0.6528
Epoch 1, Batch 10, Loss: 0.5459
Epoch 1, Batch 11, Loss: 0.5414
Epoch 1, Batch 12, Loss: 0.5361
Epoch 1, Batch 13, Loss: 0.4984
Epoch 1, Batch 14, Loss: 0.5519
Epoch 1, Batch 15, Loss: 0.3662
Epoch 1, Batch 16, Loss: 0.3869
Epoch 1, Batch 17, Loss: 0.6237
Epoch 1, Batch 18, Loss: 0.4881
Epoch 1, Batch 19, Loss: 0.5148
Epoch 1, Batch 20, Loss: 0.5135
Epoch 1, Batch 21, Loss: 0.4290
Epoch 1, Batch 22, Loss: 0.4574
Epoch 1, Batch 23, Loss: 0.4025
Epoch 1, Batch 24, Loss: 0.3460
Epoch 1, Batch 25, Loss: 0.4171
Epoch 1, Batch 26, Loss: 0.4208
Epoch 1, Batch 27, Loss: 0.2577
Epoch 1, Batch 28, Loss: 0.4442
Epoch 1, Batch 29, Loss: 0.4161
Epoch 1, Batch 30, Loss: 0.3155
Epoch 1, Batch 31, Loss: 0.3219
Epoch 1, Batch 32

In [15]:
model.eval()
all_preds = []
all_labels = []
batch_num = 0
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        batch_num += 1
        print(f"Validation Batch {batch_num} processed.")
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))

Validation Batch 1 processed.
Validation Batch 2 processed.
Validation Batch 3 processed.
Validation Batch 4 processed.
Validation Batch 5 processed.
Validation Batch 6 processed.
Validation Batch 7 processed.
Validation Batch 8 processed.
Validation Batch 9 processed.
Validation Batch 10 processed.
Validation Batch 11 processed.
Validation Batch 12 processed.
Validation Batch 13 processed.
Validation Batch 14 processed.
Validation Batch 15 processed.
Validation Batch 16 processed.
Validation Batch 17 processed.
Validation Batch 18 processed.
Validation Batch 19 processed.
Validation Batch 20 processed.
Validation Batch 21 processed.
Validation Batch 22 processed.
Validation Batch 23 processed.
Validation Batch 24 processed.
Validation Batch 25 processed.
Validation Batch 26 processed.
Validation Batch 27 processed.
Validation Batch 28 processed.
Validation Batch 29 processed.
Validation Batch 30 processed.
Validation Batch 31 processed.
Validation Batch 32 processed.
Validation Batch 

In [ ]:
""" Enlever le commentaire pour sauvegarder le modèle
import pickle
import os

# Define the directory path
output_dir = "/content/drive/MyDrive/models/"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the full path for the model file
model_path = os.path.join(output_dir, "mental_roberta_base.pkl")

# Save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)
"""